# 🔌CONEXÃO
Estabelecer conexão com os bancos

In [1]:
import psycopg2
from psycopg2.extras import DictCursor
import fdb
from dotenv import load_dotenv
import os

load_dotenv()

# Conexões
cnx_dest = fdb.connect(
    user=os.getenv("FDB_USER"),
    password=os.getenv("FDB_PASS"),
    host=os.getenv("FDB_HOST"),
    port=int(os.getenv("FDB_PORT")),
    database=os.getenv("FDB_PATH"),
    charset="WIN1252"
)
cur_dest = cnx_dest.cursor()


cnx_orig = psycopg2.connect(
    user=os.getenv("PG_USER"),
    password=os.getenv("PG_PASS"),
    host=os.getenv("PG_HOST"),
    database=os.getenv("PG_DB"),
    options="-c search_path={}".format(os.getenv("PG_SCHEMA"))
)
cnx_orig.autocommit = True
# cnx_orig.set_client_encoding('WIN1252')
cur_orig = cnx_orig.cursor(cursor_factory=DictCursor)

def commit():
    cnx_dest.commit()

# 🛠️ FERRAMENTAS
Funções, variáveis cache, hashmaps

In [2]:
global cadest, empresa, exercicio
cadest = {}
empresa = cur_dest.execute("SELECT empresa FROM cadcli").fetchone()[0]
exercicio = cur_dest.execute("SELECT mexer FROM cadcli").fetchone()[0]

def limpa_tabela(tabelas):
    for tabela in tabelas:
        cur_dest.execute(f"DELETE FROM {tabela}")
    commit()

def cria_coluna(tabela, coluna):
    try:
        cur_dest.execute(f"ALTER TABLE {tabela} ADD {coluna} VARCHAR(255)")
    except fdb.DatabaseError as e:
        print(f"Erro ao criar coluna {coluna} na tabela {tabela}: {e}")
    else:
        commit()

dict_modalidades = {
    "CC":{"Licit": "MAT / SERV - CONCORRENCIA", 
          "Modlic": "CON4",
          "Codmod": 4},
    "CV":{"Licit": "MAT / SERV - CONVITE", 
          "Modlic": "CS01",
          "Codmod": 7},
    "DE":{"Licit": "DISPENSA", 
          "Modlic": "DI01",
          "Codmod": 1},
    "DL":{"Licit": "DISPENSA", 
          "Modlic": "DI01",
          "Codmod": 1},
    "IL":{"Licit": "INEXIGIBILIDADE", 
          "Modlic": "IN01",
          "Codmod": 5},
    "PE":{"Licit": "PREGÃO ELETRÔNICO", 
          "Modlic": "PE01",
          "Codmod": 9},
    "PR":{"Licit": "PREGÃO PRESENCIAL", 
          "Modlic": "PP01",
          "Codmod": 8},
    "TP":{"Licit": "MAT / SERV - TOMADA", 
          "Modlic": "TOM3",
          "Codmod": 3},
}

def cria_coluna(tabela, coluna):
    try:
        cur_dest.execute(f"ALTER TABLE {tabela} ADD {coluna} VARCHAR(255)")
    except fdb.DatabaseError as e:
        print(f"Erro ao criar coluna {coluna} na tabela {tabela}: {e}")
    else:
        commit()

def to_cp1252_safe(value, replace_with=''):
    """
    Converte uma string para cp1252 de forma segura.
    Remove ou substitui caracteres inválidos.

    :param value: valor a converter
    :param replace_with: string usada no lugar de caracteres inválidos (default = '')
    :return: valor limpo
    """
    if isinstance(value, str):
        try:
            # Tenta converter diretamente
            value.encode('cp1252')
            return value
        except UnicodeEncodeError:
            # Remove/substitui caracteres inválidos
            return value.encode('cp1252', errors='replace').decode('cp1252').replace('?', replace_with)
    return value

def fix_mojibake(value: str) -> str:
    if isinstance(value, str):
        return value.encode("latin1").decode("utf-8")
    return value

cur_dest.execute("SELECT codreduz, max(cadpro) FROM cadest group by 1")
if cur_dest.description is None:
    print("CADEST VAZIA!")
else:
    cadest = {k:v for k,v in cur_dest.execute("SELECT codreduz, max(cadpro) FROM cadest group by 1").fetchall()}

def valida_fornecedores():
    cur_orig.execute("select identificador_de_registro codif_ant, cpf_cnpj insmf , quote_literal(substr(nome,1,50)) nome, case when tipo_da_pessoa = 'J' then '01' else '02' end codtip from pessoas where cpf_cnpj is not null")

    desfor = {k:v for v,k in cur_dest.execute("select codif, replace(replace(replace(insmf, '.',''),'-',''),'/','') insmf from desfor").fetchall()}

    for row in cur_orig.fetchall():
        codif = desfor.get(row['insmf'],0)
        if codif:
            cur_dest.execute(f"UPDATE desfor SET codif_ant = '{row['codif_ant']}' WHERE codif = '{codif}'")
        else:
            cur_dest.execute(f"INSERT INTO desfor (codif, insmf, nome, codtip, codif_ant) VALUES ({row['codif_ant']}, '{row['insmf']}', {row['nome']}, '{row['codtip']}', '{row['codif_ant']}')")
    commit()
# valida_fornecedores()

# 🛒 COMPRAS
<p>Extração, tratamento e carregamento dos dados referentes ao módulo compras</p>

In [ ]:
cur_dest.execute("""
execute block as
		begin
		DELETE FROM ICADREQ;
		DELETE FROM REQUI;
		DELETE FROM ICADPED;
		DELETE FROM CADPED;
		DELETE FROM regpreco;
		DELETE FROM regprecohis;
		DELETE FROM regprecodoc;
		DELETE FROM CADPRO_SALDO_ANT;
		DELETE FROM CADPROLIC_DETALHE_FIC;
		DELETE FROM CADPRO;
		DELETE FROM CADPRO_FINAL;
		DELETE FROM CADPRO_LANCE;
		DELETE FROM CADPRO_PROPOSTA;
		DELETE FROM PROLICS;
		DELETE FROM PROLIC;
		DELETE FROM CADPRO_STATUS;
		DELETE FROM CADLIC_SESSAO;
		DELETE FROM CADPROLIC_DETALHE;
		DELETE FROM CADPROLIC;
		DELETE FROM CADLIC;
		DELETE FROM VCADORC;
		DELETE FROM FCADORC;
		DELETE FROM ICADORC;
		DELETE FROM CADORC;
		DELETE FROM CADEST;
		DELETE FROM CENTROCUSTO;
		DELETE FROM DESTINO;
		DELETE FROM DESFORCRC_PADRAO;
		end;
""")
commit()

## CADASTROS BASE

### CADUNIMEDIDA

In [ ]:
limpa_tabela(("cadunimedida",))
cria_coluna("cadunimedida", "codant")

insert = cur_dest.prep("insert into cadunimedida(sigla,descricao,codant) values(?,?,?)")

cur_orig.execute("""
WITH base AS (
    SELECT 
        trim(SUBSTR(s_mbolo_da_unidade_de_medida, 1, 5)) AS sigla_base,
        nome_da_unidade_de_medida AS descricao,
        c_digo_da_unidade_de_medida AS codant,
        ROW_NUMBER() OVER (
            PARTITION BY trim(SUBSTR(s_mbolo_da_unidade_de_medida, 1, 5))
            ORDER BY c_digo_da_unidade_de_medida
        ) AS ordem
    FROM unidades_medida
)
SELECT 
    trim(CASE 
        WHEN ordem = 1 THEN sigla_base
        ELSE SUBSTR(sigla_base, 1, 5 - LENGTH(ordem::text)) || ordem::text
    END) AS sigla,
    descricao,
    codant
FROM base
ORDER BY sigla_base, ordem
""")

repetidos = {}

try:
    for row in cur_orig:
        sigla = to_cp1252_safe(row['sigla'])
        descricao = row['descricao']
                    
        repetidos[sigla] = repetidos.get(sigla, 0) + 1
        cur_dest.execute(insert, (sigla, row['descricao'], row['codant']))
    commit()
except Exception as e:
    print(f"Erro ao inserir unidade de medida: {e} - {row}")

### GRUPO E SUBGRUPO

In [ ]:
limpa_tabela(("cadsubgr", "cadgrupo"))
cria_coluna("cadgrupo", "codant")
cria_coluna("cadsubgr", "codant")

cur_orig.execute("""
select
	to_char(dense_rank() over (order by gdm.identificador_de_registro), 'fm000') grupo,
	'000' subgrupo,
	upper(descricao) nome,
	gdm.identificador_de_registro codant
from
	grupos_de_materiais gdm
union all
select
	grupo,
	to_char(dense_rank() over (partition by cdp.codigo_do_grupo order by cdp.identificador_do_registro), 'fm000') subgrupo,
	upper(descricao) nome,
	cdp.identificador_do_registro codant
from
	classe_dos_produtos cdp
join (
select
	to_char(dense_rank() over (order by gdm.identificador_de_registro), 'fm000') grupo,
	gdm.identificador_de_registro
from
	grupos_de_materiais gdm) grp on
cdp.codigo_do_grupo = grp.identificador_de_registro 
order by 2,1
""")

for row in cur_orig:
	try:
		if row['subgrupo'] == '000':
			cur_dest.execute("insert into cadgrupo(grupo, nome, codant) values(?, ?, ?)", (row['grupo'], row['nome'][:45], row['codant']))
		else:
			cur_dest.execute("insert into cadsubgr(grupo, subgrupo, nome, codant) values(?, ?, ?, ?)", (row['grupo'], row['subgrupo'], row['nome'][:45], row['codant']))
		commit()
	except Exception as e:
		print(f"Erro ao inserir grupo/subgrupo {row['grupo']}/{row['subgrupo']}: {e} - {row.items()}")


cur_dest.execute("insert into cadgrupo(grupo, nome) values ('000', 'SEM IDENTIFICAÇÃO')")
commit()

cur_dest.execute("insert into cadsubgr(grupo, subgrupo, nome) values ('000', '000', 'SEM IDENTIFICAÇÃO')")
commit()

### CADEST

In [ ]:
limpa_tabela(("cadest",))
cria_coluna("cadest", "key")
cria_coluna("cadest", "codant")

insert = cur_dest.prep("insert into cadest(grupo, subgrupo, codigo, cadpro, codreduz, disc1, ocultar, unid1, tipopro, usopro, codant, key) values (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)")

cur_orig.execute("""
with UNIDMEDIDAS as (
WITH base AS (
    SELECT 
        trim(SUBSTR(s_mbolo_da_unidade_de_medida, 1, 5)) AS sigla_base,
        nome_da_unidade_de_medida AS descricao,
        c_digo_da_unidade_de_medida AS codant,
        ROW_NUMBER() OVER (
            PARTITION BY trim(SUBSTR(s_mbolo_da_unidade_de_medida, 1, 5))
            ORDER BY c_digo_da_unidade_de_medida
        ) AS ordem
    FROM unidades_medida
)
SELECT 
    trim(CASE 
        WHEN ordem = 1 THEN sigla_base
        ELSE SUBSTR(sigla_base, 1, 5 - LENGTH(ordem::text)) || ordem::text
    END) AS sigla,
    descricao,
    codant
FROM base
ORDER BY sigla_base, ordem),
SUBGRUPOS as (
select
	grupo,
	to_char(dense_rank() over (partition by cdp.codigo_do_grupo order by cdp.identificador_do_registro), 'fm000') subgrupo,
	upper(descricao) nome,
	cdp.identificador_do_registro codant
from
	classe_dos_produtos cdp
join (
select
	to_char(dense_rank() over (order by gdm.identificador_de_registro), 'fm000') grupo,
	gdm.identificador_de_registro
from
	grupos_de_materiais gdm) grp on
cdp.codigo_do_grupo = grp.identificador_de_registro 
order by 2,1)
select
	s.grupo,
	s.subgrupo,
	m.descricao disc1,
	codigo_do_material_1 codant,
	sigla unid1,
	case when ativo = 'S' then 'N' else 'S' end ocultar,
	tipo,
	codigo_do_material codreduz
from
	materiais m
join UNIDMEDIDAS u on m.codigo_da_unidade_de_estoque = u.codant
join SUBGRUPOS s on s.codant = m.codigo_da_classe
""")

subgrupos_desdobrados = {k:[v, 0] for k, v in cur_dest.execute("select grupo||'.'||subgrupo, cast(subgrupo as integer) subgrupo_int from cadsubgr")}
max_subgrupo_grupo = {k: v for k, v in cur_dest.execute("select grupo, max(cast(subgrupo as integer)) from cadsubgr group by 1")}

tipo_uso_produto = {
	'M': ['P', 'C'],
	'P': ['P', 'P'],
	'S': ['S', '']
}

for row in cur_orig:
	key = f"{row['grupo']}.{row['subgrupo']}"
	subgrupo_int, codigo = subgrupos_desdobrados[key]
	codigo += 1

	if codigo > 999:
		subgrupo_ant = max_subgrupo_grupo[row['grupo']]
		subgrupo_int = max_subgrupo_grupo[row['grupo']] + 1
		codigo = 1
		subgrupos_desdobrados[key] = [subgrupo_int, codigo]
		max_subgrupo_grupo[row['grupo']] = subgrupo_int
	else:
		subgrupos_desdobrados[key][1] = codigo

	tipopro, usopro = tipo_uso_produto.get(row['tipo'], ['P', 'C'])

	cadpro = f'{row['grupo']}.{subgrupo_int:03}.{codigo:03}'

	disc1 = to_cp1252_safe(row['disc1'], ' ')[:1024]

	cur_dest.execute(insert, (row['grupo'], f'{subgrupo_int:03}', f'{codigo:03}', cadpro, row['codreduz'], disc1, row['ocultar'], row['unid1'], tipopro, usopro, row['codant'], key))
commit()

cur_dest.execute("""
INSERT INTO CADSUBGR (grupo, subgrupo, nome)
WITH CTE AS (
SELECT DISTINCT a.grupo, a.subgrupo, KEY FROM cadest a
WHERE NOT EXISTS (SELECT 1 FROM cadsubgr x WHERE a.grupo = x.grupo AND a.SUBGRUPO = x.SUBGRUPO))
SELECT a.GRUPO, a.SUBGRUPO, b.NOME FROM cte a
JOIN CADSUBGR b ON b.grupo||'.'||b.SUBGRUPO = a."KEY" 
""")
commit()

cadest = {k: v for k,v in cur_dest.execute("select codreduz, max(cadpro) from cadest group by codreduz").fetchall()}

### CENTRO DE CUSTO

In [ ]:
limpa_tabela(("centrocusto",))
cria_coluna("centrocusto", "codant")

insert = cur_dest.prep("""
    insert into centrocusto (codccusto, descr, ccusto, ocultar, empresa, poder, orgao, destino, codant, obs) values (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

cur_orig.execute("""
with centrocusto as (
select
	oc.numero_que_representa_o_organograma codant,
	max(oc.descricao) descr,
	max(codigo_dos_organogramas_contratos) id
from
	organogramas_contratos oc
group by
	1)
select
	substr(codant, 1, 2) poder,
	substr(codant, 4, 2) orgao,
	--substr(codant,9,2) unidade,
	dense_rank() over (order by id) codccusto,
	codant,
	descr,
	'000000001' destino
from
	centrocusto
""")

for row in cur_orig:
    cur_dest.execute(insert, (f'{empresa}{row['codccusto']}', row['descr'], '001', 'N', empresa, row['poder'], row['orgao'], row['destino'], row['codant'], str(row['codccusto'])))
cur_dest.execute(insert, (f'{empresa}0', 'CONVERSÃO', '001', 'N', empresa, '02', '01', '000000001', None, '0'))
commit()

# Replica centrocusto para outras empresas
cur_dest.execute("""
INSERT
	INTO
	centrocusto (codccusto,
	descr,
	ccusto,
	ocultar,
	empresa,
	poder,
	orgao,
	destino,
	codant)
WITH cte AS (	
SELECT
	b.empresa||obs codccusto,
	codccusto pai,
	descr,
	ccusto,
	ocultar,
	b.empresa,
	a.poder,
	a.orgao,
	destino,
	codant
FROM
	centrocusto a,
	tabempresa b)
SELECT codccusto,
	descr,
	ccusto,
	ocultar,
	empresa,
	poder,
	orgao,
	lpad(empresa, 9,'0') destino,
	codant FROM cte
WHERE NOT EXISTS (SELECT 1 FROM centrocusto c WHERE c.codccusto = cte.pai AND c.EMPRESA = cte.EMPRESA)
""")
commit()



## COTAÇÃO



In [ ]:
cur_dest.execute("""
execute block as
    begin
    DELETE FROM vcadorc;
    DELETE FROM fcadorc;
    DELETE FROM icadorc;
    DELETE FROM cadorc;
end;
""")
commit()

### CADORC

In [ ]:
limpa_tabela(("cadorc",))

insert = cur_dest.prep("""
insert into cadorc
	(id_cadorc,
	num,
	ano,
	numorc,
	dtorc,
	descr,
	prioridade,
	obs,
	status,
	liberado,
	codccusto,
	liberado_tela,
	empresa) values
	(?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

cur_orig.execute("""
with BASE as (with centrocusto as (
select
	oc.numero_que_representa_o_organograma codant,
	max(oc.descricao) descr,
	max(codigo_dos_organogramas_contratos) id
from
	organogramas_contratos oc
group by
	1)
select
	substr(codant, 1, 2) poder,
	substr(codant, 4, 2) orgao,
	--substr(codant,9,2) unidade,
	dense_rank() over (order by id) codccusto,
	codant,
	descr,
	'000000001' destino
from
	centrocusto)
select distinct
	cadorc.identificador_de_registro::int id_cadorc,
	to_char(cadorc.numero_da_cotacao, 'fm00000') num,
	p.exercicio ano,
	concat(to_char(cadorc.numero_da_cotacao, 'fm00000/'), p.exercicio%2000) numorc,
	data_da_cotacao::date dtorc,
	objeto descr,
	max(coalesce(observacoes, assunto)) obs,
	case when situacao = 'C' then 'CO' when situacao = 'F' then 'EC' else 'CA' end status,
	case when situacao = 'F' then 'S' else 'N' end liberado,
	max(concat(cadorc.codigo_da_entidade::int,coalesce(codccusto,0)::int)) codccusto,
	cadorc.codigo_da_entidade::int empresa
from
	cotacoes_de_preco cadorc
join parametros_por_exercicio p on p.codigo_sequencial = cadorc.codigo_sequencial
left join solicitacoes_x_cotacoes sxc on cadorc.identificador_de_registro = sxc.identificador_de_registro
left join solicitacoes sol on sol.identificador_das_solicitacoes = sxc.identificador_das_solicitacoes
left join organogramas_contratos oc on sol.codigo_dos_organograma = oc.codigo_dos_organogramas_contratos
left join BASE on BASE.codant = oc.numero_que_representa_o_organograma
--where cadorc.codigo_da_entidade = 538
group by 1,2,3,4,5,6,8,9,11
order by 1
""")

for row in cur_orig:
	cur_dest.execute(insert, (
		row['id_cadorc'],
		row['num'],
		row['ano'],
		row['numorc'],
		row['dtorc'],
		row['descr'][:1024],
		'NORMAL',
		row['obs'],
		row['status'],
		row['liberado'],
		row['codccusto'],
		None,
		row['empresa']
	))
commit()

### ICADORC

In [ ]:
limpa_tabela(("icadorc",))

insert = cur_dest.prep("""
insert into icadorc (
    numorc,
    item,
    cadpro,
    qtd, 
    valor, 
    itemorc, 
    codccusto, 
    itemorc_ag,
    id_cadorc)
values(?,?,?,?,?,?,?,?,?)
""")

cur_orig.execute("""with BASE as (with centrocusto as (
select
	oc.numero_que_representa_o_organograma codant,
	max(oc.descricao) descr,
	max(codigo_dos_organogramas_contratos) id
from
	organogramas_contratos oc
group by
	1)
select
	substr(codant, 1, 2) poder,
	substr(codant, 4, 2) orgao,
	--substr(codant,9,2) unidade,
	dense_rank() over (order by id) codccusto,
	codant,
	descr,
	'000000001' destino
from
	centrocusto)
select distinct
	concat(to_char(cadorc.numero_da_cotacao, 'fm00000/'), p.exercicio%2000) numorc,
	idc.numero_do_item::int item,
	idc.codigo_do_material codreduz,
	idc.quantidade_de_itens_solicitados qtd,
	0 valor,
	idc.identificador_de_registro_da_cotacao id_cadorc,
	idc.codigo_da_entidade entidade
from
	itens_das_cotacoes idc
join cotacoes_de_preco cadorc on idc.identificador_de_registro_da_cotacao = cadorc.identificador_de_registro 
join parametros_por_exercicio p on p.codigo_sequencial = cadorc.codigo_sequencial
left join solicitacoes_x_cotacoes sxc on cadorc.identificador_de_registro = sxc.identificador_de_registro
left join solicitacoes sol on sol.identificador_das_solicitacoes = sxc.identificador_das_solicitacoes
left join organogramas_contratos oc on sol.codigo_dos_organograma = oc.codigo_dos_organogramas_contratos
left join BASE on BASE.codant = oc.numero_que_representa_o_organograma
--where idc.codigo_da_entidade = 538
order by identificador_de_registro_da_cotacao, 2""")

for row in cur_orig:
    try:
        key = str(row['codreduz'])
        cadpro = cadest[key]
    except Exception as e:
        print(f"Erro ao obter cadpro: {e}")
        break

    else:
        cur_dest.execute(insert, (
            row['numorc'],
            row['item'],
            cadpro,
            row['qtd'],
            row['valor'],
            row['item'],
            0,
            row['item'],
            row['id_cadorc']
        ))
commit()

cur_dest.execute("update icadorc a set a.codccusto = (select codccusto from cadorc b where a.id_cadorc = b.id_cadorc)")
commit()
    

### FCADORC

In [ ]:
limpa_tabela(("fcadorc",))

insert = cur_dest.prep("insert into fcadorc(numorc, codif, nome, valorc, id_cadorc) values (?,?,?,?,?)")

cur_orig.execute("""
select
	concat(to_char(numero_da_cotacao, 'fm00000/'), p.exercicio%2000) numorc,
	a.codigo_da_pessoa::int codif,
	d.nome,
	valorc,
	a.identificador_de_registro id_cadorc,
	c.codigo_da_entidade empresa
from
	participantes_das_cotacoes a
join (select identificador_de_registro_participantes, sum(valor_cotado) valor, sum(valor_total) valorc  from valores_cotados_com_os_fornecedores vccof group by 1) b
using (identificador_de_registro_participantes) 
join cotacoes_de_preco c on c.identificador_de_registro = a.identificador_de_registro
join parametros_por_exercicio p on p.codigo_sequencial = c.codigo_sequencial
left join pessoas d on a.codigo_da_pessoa = d.identificador_de_registro
--where c.codigo_da_entidade = 538
order by 5, 2
""")

for row in cur_orig:
	cur_dest.execute(insert, (
		row['numorc'],
		row['codif'],
		row['nome'][:70],
		row['valorc'],
		row['id_cadorc']
	))
commit()

### VCADORC

In [ ]:
limpa_tabela(("vcadorc",))
cria_coluna("vcadorc", "tipo")

insert = cur_dest.prep("""insert into vcadorc(numorc, item, codif, vlruni, vlrtot, ganhou, vlrganhou, classe, id_cadorc, tipo) values (?,?,?,?,?,?,?,?,?,?)""")

cur_orig.execute("""
with ganhadores as (
select
    identificador_de_registro,
    idc.numero_do_item item,
    pdc.codigo_da_pessoa ganhou,
    valor_cotado vlrganhou,
    pdc.codigo_da_entidade empresa,
    row_number() over (partition by identificador_de_registro, idc.numero_do_item order by valor_cotado asc) seq
from
    participantes_das_cotacoes pdc
join (select *  from valores_cotados_com_os_fornecedores) vccof using (identificador_de_registro_participantes) 
join cotacoes_de_preco cdp using(identificador_de_registro)
join itens_das_cotacoes idc on idc.identificador_do_itens_da_cotacao = vccof.identificador_do_itens_da_cotacao 
where vccof.situacao = 1
order by 2, 3)
select
    concat(to_char(numero_da_cotacao, 'fm00000/'), extract(year from data_da_cotacao)::int%2000) numorc,
    idc.numero_do_item::int item,
    pdc.codigo_da_pessoa::int codif,
    valor_cotado vlruni,
    vccof.valor_total vlrtot,
    vlrganhou,
    ganhou::int,
    pdc.identificador_de_registro id_cadorc,
    idc.media_saneada_do_item_da_cotacao vlruni_medio,
    idc.valor_total vlrtot_medio,
    case
    when ganhou = pdc.codigo_da_pessoa then case
        when cdp.tipo_do_preco_medio_mediano_ou_melhor = 1 then 'M'
        when cdp.tipo_do_preco_medio_mediano_ou_melhor = 2 then 'V'
        when cdp.tipo_do_preco_medio_mediano_ou_melhor = 3 then 'D'		
        when cdp.tipo_do_preco_medio_mediano_ou_melhor = 4 then 'S'		
    end 
    end tipo
    --1 médio 2 melhor 3 mediano 4 saneada
from
    participantes_das_cotacoes pdc
join (select *  from valores_cotados_com_os_fornecedores) vccof using (identificador_de_registro_participantes) 
join cotacoes_de_preco cdp using(identificador_de_registro)
join itens_das_cotacoes idc on idc.identificador_do_itens_da_cotacao = vccof.identificador_do_itens_da_cotacao 
left join ganhadores g on g.identificador_de_registro = pdc.identificador_de_registro and item = idc.numero_do_item and empresa = pdc.codigo_da_entidade and seq = 1
--where cdp.codigo_da_entidade = 538
where case
    when ganhou = pdc.codigo_da_pessoa then case
        when cdp.tipo_do_preco_medio_mediano_ou_melhor = 1 then 'M'
        when cdp.tipo_do_preco_medio_mediano_ou_melhor = 2 then 'V'
        when cdp.tipo_do_preco_medio_mediano_ou_melhor = 3 then 'D'		
        when cdp.tipo_do_preco_medio_mediano_ou_melhor = 4 then 'S'		
    end 
    end is not null
order by 2, 3
""")

for row in cur_orig:
    try:
        cur_dest.execute(insert, (
            row['numorc'],
            row['item'],
            row['codif'],
            row['vlruni'],
            row['vlrtot'],
            row['ganhou'],
            row['vlrganhou'],
            'UN',
            row['id_cadorc'],
            row['tipo']
        ))
    except Exception as e:
        print(f"Erro ao inserir vcadorc: {e} - {row}")
        break
commit()

## LICITAÇÕES

In [ ]:
#LIMPA LICITAÇÕES
cur_dest.execute("""
execute block as
    begin
    DELETE FROM regpreco;
    DELETE FROM regprecohis;
    DELETE FROM regprecodoc;
    DELETE FROM CADPROLIC_DETALHE_FIC;
    DELETE FROM CADPRO;
    DELETE FROM CADPRO_FINAL;
    DELETE FROM CADPRO_LANCE;
    DELETE FROM CADPRO_PROPOSTA;
    DELETE FROM PROLICS;
    DELETE FROM PROLIC;
    DELETE FROM CADPRO_STATUS;
    DELETE FROM CADLIC_SESSAO;
    DELETE FROM CADPROLIC_DETALHE;
    DELETE FROM CADPROLIC;
    DELETE FROM CADLIC;
    end;
""")
commit()

### CADLIC

In [ ]:
limpa_tabela(("cadlic",))
cria_coluna("cadlic", "tp_origem")
cria_coluna("cadlic", "tpcontrole_saldo")

insert = cur_dest.prep("""
INSERT
	INTO
	cadlic(
	numlic,
	proclic,
	numero,
	ano,
	comp,
	licnova,
	liberacompra,
	discr,
	detalhe,
	registropreco,
	microempresa,
	numpro,
	discr7,
	datae,
	processo_data,
	dtadj,
	dthom,
	codtce,
	anomod,
	modlic,
	licit,
	codmod,
	dtpub,
	dtenc,
	valor,
	empresa,
	processo,
	processo_ano,
	dtreal,
	numorc,
	id_cadorc,
	tpapostilamento,
    tpcontrole_saldo,
	dtpropostaini,
	dtpropostafim,
	tlance,
	tp_origem)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

cur_orig.execute("""
with cadorc as (select
	identificador_dos_processos_administrativos numlic,
	max(s.codigo_dos_organograma) codigo_dos_organograma,
	max(coalesce(cp.identificador_de_registro,s.identificador_das_solicitacoes)) id_cadorc,
	max(concat(to_char(coalesce(numero_da_cotacao::text,substr(s.codigo_sequencial_das_solicitacoes::text,1,5))::int,'fm00000/'),coalesce(exc.exercicio, ex.exercicio)%2000)) numorc,
	case when cp.identificador_de_registro is not null then 'C' else 'S' end tipo_origem
from
	processo_de_compra_x_solicitacao pxs
join solicitacoes s on pxs.identificador_das_solicitacoes = s.identificador_das_solicitacoes 
join parametros_por_exercicio ex on ex.codigo_sequencial = s.codigo_sequencial 
left join solicitacoes_x_cotacoes sxc on s.identificador_das_solicitacoes = sxc.identificador_das_solicitacoes 
left join cotacoes_de_preco cp on cp.identificador_de_registro = sxc.identificador_de_registro 
left join parametros_por_exercicio exc on exc.codigo_sequencial = cp.codigo_sequencial
group by 1,5)
select
	pa.identificador_dos_processos_administrativos::int numlic,
	concat(to_char(pa.codigo_sequencial_das_solicitacoes, 'fm000000/'),exercicio%2000) proclic,
	pa.codigo_sequencial_das_solicitacoes::int numero,
	exercicio::int ano,
	case when pa.status_do_processo = 11 then 3 when pa.status_do_processo = 0 then 1 else 2 end comp,
	1 licnova,
	case when pa.status_do_processo = 11 then 'S' else 'N' end liberacompra,
	pa.objeto_do_processo discr,
	tdo.descricao detalhe,
	registro_de_preco registropreco,
	3 microempresa,
	numeracao_forma_contratacao::int numpro,
	/*fdj.descricao*/ 'Menor Preco Unitario' discr7,
	pa.data_do_processo::date datae,
	data_do_processo::date processo_data,
	paaf.data_e_hora_do_ato::date dtajd,
	paaf.data_e_hora_do_ato::date dthom,
	null codtce,
	c.ano_sequencial::int anomod,
	mdl.sigla modlic,
	null licit,
	null codmod,
	pdp.data_da_publicacao::date dtpub,
	paaf.data_e_hora_do_ato::date dtenc,
	pa.valor_total_dos_itens valor,
	pa.codigo_da_entidade::int empresa,
	numero_protocolo processo,
	pa.ano_do_protocolo processo_ano,
	data_hora_abertura_da_sessao_julgamento::date dtreal,
	data_e_hora_abertura_envelopes::date dtenv,
	numorc,
	id_cadorc::int,
	case when pa.controle_de_saldo = 'V' then 'T' else controle_de_saldo end tpcontrole_saldo,
	data_e_hora_inicio_recebimento_envelopes::date dtpropostaini,
	data_e_hora_final_recebimento_envelopes::date dtpropostafim,
	coalesce(tipo_origem, 'D') tipo_origem
from
	processos_administrativos pa
join parametros_por_exercicio pe on pe.codigo_sequencial = pa.codigo_sequencial
left join formas_de_julgamento fdj on fdj.identificador_do_tipo_do_objeto = pa.identificador_da_forma_julgamento 
left join tipos_de_objetos tdo on tdo.identificador_do_tipo_do_objeto = pa.identificador_do_tipo_do_objeto
left join (select identificador_dos_processos_administrativos, max(data_da_publicacao) data_da_publicacao from publicacoes_do_processo group by 1) pdp on pdp.identificador_dos_processos_administrativos = pa.identificador_dos_processos_administrativos 
left join cadorc on cadorc.numlic = pa.identificador_dos_processos_administrativos
left join contratacoes c on c.identificador_dos_processos_administrativos = pa.identificador_dos_processos_administrativos
left join modalidades_de_licitacao mdl on c.identificador_de_registro = mdl.identificador_de_registro
left join processos_administrativos_atos_finais paaf on paaf.identificador_dos_processos_administrativos = pa.identificador_dos_processos_administrativos and tipo_do_ato = 3
left join (select row_number() over (partition by identificador_dos_processos_administrativos_contratacoes order by identificador_das_sessoes_de_julgamento) seq, * from processos_administrativos_sessoes_de_julgamento) pasdj 
on pasdj.identificador_dos_processos_administrativos_contratacoes = pa.identificador_dos_processos_administrativos and seq = 1
--where pa.codigo_da_entidade = 538
and numeracao_forma_contratacao is not null
order by 1
""")

for row in cur_orig:
    info_modalidades = dict_modalidades.get(row['modlic'], {"Licit": "DISPENSA", "Modlic": "DI01", "Codmod": 1})

    licit = info_modalidades["Licit"]
    modlic = info_modalidades["Modlic"]
    codmod = info_modalidades["Codmod"]

    cur_dest.execute(insert, (
		row['numlic'], 
		row['proclic'],
		row['numero'],
		row['ano'],
		row['comp'],
		row['licnova'],
		row['liberacompra'],
		to_cp1252_safe(row['discr']),
		to_cp1252_safe(row['detalhe']),
		row['registropreco'],
		row['microempresa'],
		row['numpro'],
		to_cp1252_safe(row['discr7']),
		row['datae'],
		row['processo_data'],
		row['dthom'],
		row['dthom'],
		row['codtce'],
		row['ano'],
		modlic,
		licit,
		codmod,
		row['dtpub'],
		row['dthom'],
		row['valor'],
		row['empresa'],
		row['processo'],
		row['processo_ano'],
		row['datae'],
		row['numorc'],
		row['id_cadorc'],
        'I',
        row['tpcontrole_saldo'],
        row['dtpropostaini'],
		row['dtpropostafim'],
        '$',
        row['tipo_origem']
	))
commit()

cur_dest.execute("""EXECUTE BLOCK AS
DECLARE VARIABLE DESCMOD VARCHAR(1024);
DECLARE VARIABLE CODMOD INTEGER;
BEGIN
    FOR
        SELECT CODMOD, DESCMOD FROM MODLIC INTO :CODMOD, :DESCMOD
    DO
    BEGIN
        UPDATE CADLIC SET LICIT = :DESCMOD where CODMOD = :CODMOD;
    END
END""")

cur_dest.execute("""UPDATE CADLIC SET FK_MODLICANO = (SELECT PK_MODLICANO FROM MODLICANO WHERE CODMOD = CADLIC.CODMOD AND ANOMOD = CADLIC.ANO AND CADLIC.EMPRESA = MODLICANO.EMPRESA) WHERE CODMOD IS NOT NULL""")
commit()

cur_dest.execute("alter trigger TBU_CADLICITACAO inactive")
commit()

cur_dest.execute("UPDATE cadlicitacao SET nlicitacao = nlicitacao * 100000")
commit()

cur_dest.execute("UPDATE cadlicitacao SET nlicitacao = numlic")
commit()

cur_dest.execute("MERGE INTO cadlic a USING (select id_cadorc, numorc FROM cadorc ) b ON a.id_cadorc = b.id_cadorc WHEN MATCHED THEN UPDATE SET a.numorc = b.numorc")
commit()

cur_dest.execute("""MERGE
INTO
	cadorc a
		USING (
	SELECT
		id_cadorc,
		max(numlic) numlic,
		max(proclic) proclic
	FROM
		cadlic
	GROUP BY
		1) b
ON
	a.id_cadorc = b.id_cadorc
	WHEN MATCHED THEN
UPDATE
SET
	a.liberado = 'S',
	a.liberado_tela = 'L',
	a.status = 'LC',
	a.numlic = b.numlic,
	a.proclic = b.proclic""")
commit()

### CADLIC SESSAO

In [ ]:
cur_dest.execute("""INSERT INTO CADLIC_SESSAO (NUMLIC, SESSAO, DTREAL, HORREAL, COMP, DTENC, HORENC, SESSAOPARA, MOTIVO) 
SELECT L.NUMLIC, CAST(1 AS INTEGER), L.DTREAL, L.HORREAL, L.COMP, L.DTENC, L.HORENC, CAST('T' AS VARCHAR(1)), CAST('O' AS VARCHAR(1)) FROM CADLIC L 
WHERE numlic not in (SELECT FIRST 1 S.NUMLIC FROM CADLIC_SESSAO S WHERE S.NUMLIC = L.NUMLIC)""")
commit()

### PARTICIPANTES

In [ ]:
limpa_tabela(("prolics","prolic"))

insert_prolic = cur_dest.prep("insert into prolic(numlic,codif,nome,status,controle_ata_rp,dtprazo) values(?,?,?,?,?,?)")
insert_prolics = cur_dest.prep("insert into prolics(sessao,numlic,codif,habilitado,status,cpf,representante) values(?,?,?,?,?,?,?)")

cur_orig.execute("""
with atas as (select
	identificador_dos_processos_administrativos_1 numlic,
	identificador_do_do_fornecedor codif,
	concat(arp.numero_da_ata, '/', extract(year from data_da_ata)) controle_ata_rp,
	data_de_vencimento_da_ata::date dtprazo,
	row_number() over (partition by identificador_do_processo_administrativo, identificador_do_do_fornecedor order by identificador_da_ata desc) seq
from
	atas_regist_preco arp
join cache_dos_processos_administrativos_e_contratacoes c on arp.identificador_do_processo_administrativo = c.identificador_dos_processos_administrativos
--where identificador_da_entidade = 538
)
select distinct
	pdl.identificador_dos_processos_administrativos::int numlic,
	pdl.codigo_da_pessoa::int codif,
	substr(p.nome,1,40) nome,
	cpf_cnpj cpf,
	case when situacao_da_documentacao = 'H' then 'S' else situacao_da_documentacao end status,
	controle_ata_rp::int,
	dtprazo
from
	participantes_da_licitacao pdl
join pessoas p on pdl.codigo_da_pessoa = p.identificador_de_registro 
left join atas on atas.numlic = pdl.identificador_dos_processos_administrativos and atas.codif = pdl.codigo_da_pessoa and seq = 1
""")

for row in cur_orig:
	try:
		cur_dest.execute(insert_prolic, (row['numlic'], row['codif'], to_cp1252_safe(row['nome']), row['status'], row['controle_ata_rp'], row['dtprazo']))
		cur_dest.execute(insert_prolics, (1, row['numlic'], row['codif'], row['status'], row['status'], row['cpf'], to_cp1252_safe(row['nome'])))
	except Exception as e:
		print(f"Error : {e} - {row}")
commit()

cur_dest.execute("""
MERGE INTO desfor a using (
SELECT DISTINCT b.CODIF, a.codif codif_ant, c.NOME FROM prolics a 
JOIN desfor b ON a.CPF = replace(replace(REPLACE(b.INSMF,'.',''),'-',''),'/','')
JOIN prolic c ON c.numlic = a.numlic AND c.codif = a.CODIF 
WHERE c.FK_CADLIC IS NOT NULL) b
ON a.codif = b.codif
WHEN MATCHED THEN UPDATE SET a.codif_ant = b.codif_ant
""")
commit()

cur_dest.execute("""
MERGE INTO prolics a USING (SELECT codif, codif_ant FROM desfor) b
ON a.codif = b.codif_ant
WHEN MATCHED THEN UPDATE SET a.codif = b.codif
""")
commit()

cur_dest.execute("""
MERGE INTO prolic a USING (SELECT codif, codif_ant FROM desfor) b
ON a.codif = b.codif_ant
WHEN MATCHED THEN UPDATE SET a.codif = b.codif
""")
commit()

### CADPROLIC

In [ ]:
limpa_tabela(("cadprolic_detalhe_fic", "cadprolic_detalhe", "cadprolic"))
cria_coluna("cadprolic", "tp_origem")

insert_cadprolic = cur_dest.prep("""
INSERT INTO
		cadprolic(item,
		item_mask,
		numorc,
		id_cadorc,
		itemorc,
		cadpro,
		quan1,
		vamed1,
		vatomed1,
		codccusto,
		reduz,
		numlic,
        microempresa,
	    item_lc147,
		tp_origem)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

cur_orig.execute("""
select distinct
	ipag.numero_do_item::int item,
	ipag.numero_do_item::int item_mask,
	concat(to_char(coalesce(cdp.numero_da_cotacao,s.codigo_sequencial_das_solicitacoes), 'fm00000/'), coalesce(p.exercicio, ps.exercicio)%2000) numorc,
	coalesce(sxc.identificador_de_registro, s.identificador_das_solicitacoes)::int id_cadorc,
	--idsdc.numero_do_item::int itemorc,
	idpa.codigo_do_material codreduz,
	ipag.quantidade_do_item_do_processo quan1,
	ipag.valor_total / ipag.quantidade_do_item_do_processo vamed1,
	ipag.valor_total vatomed1,
	concat(idpa.codigo_da_entidade, 0)::int codccusto,
	'N' microempresa,
	idpa.identificador_dos_processos_administrativos::int numlic,
	case when cdp.numero_da_cotacao is not null then 'C' when s.codigo_sequencial_das_solicitacoes is not null then 'S' else 'D' end tipo_origem
from
	itens_do_processo_administrativo idpa --24886
join itens_do_processo_administrativo_gerados ipag on
	idpa.identificador_do_item_do_processo_administrativo = ipag.item_principal
left join itens_do_processo_de_compra_x_itens_da_solicitacao ipxs on idpa.identificador_do_item_do_processo_administrativo = ipxs.identificador_dos_processos_administrativos  
left join itens_das_solicitacoes_de_compra idsdc on idsdc.identificador_unico_dos_itens = ipxs.identificador_unico_dos_itens
left join solicitacoes_x_cotacoes sxc on sxc.identificador_das_solicitacoes = idsdc.codigo_que_referencia_o_id_da_tabela_de_solicitacoes
left join cotacoes_de_preco cdp on cdp.identificador_de_registro = sxc.identificador_de_registro
left join parametros_por_exercicio p on p.codigo_sequencial = cdp.codigo_sequencial
left join processo_de_compra_x_solicitacao pxs on pxs.identificador_dos_processos_administrativos = idpa.identificador_dos_processos_administrativos
left join solicitacoes s on s.identificador_das_solicitacoes = idsdc.codigo_que_referencia_o_id_da_tabela_de_solicitacoes
left join parametros_por_exercicio ps on ps.codigo_sequencial = s.codigo_sequencial
--where idpa.codigo_da_entidade = 538 
order by numlic, item
""")

for row in cur_orig:
    try:
        cadpro = cadest[str(row['codreduz'])]
    except KeyError:
        continue

    cur_dest.execute(insert_cadprolic, (row['item'], row['item'], row['numorc'], row['id_cadorc'], None, cadpro, row['quan1'], row['vamed1'], row['vatomed1'], row['codccusto'], 'N', row['numlic'], row['microempresa'], None, row['tipo_origem']))
commit()

cur_dest.execute("""
	INSERT INTO CADPROLIC_DETALHE (NUMLIC,item,CADPRO,quan1,VAMED1,VATOMED1,marca,CODCCUSTO,ITEM_CADPROLIC, item_lc147)
	select numlic, item, cadpro, quan1, vamed1, vatomed1, marca, codccusto, item, item_lc147 from cadprolic b where
	not exists (select 1 from cadprolic_detalhe c where b.numlic = c.numlic and b.item = c.item);
""")
commit()

cur_dest.execute("""
	insert into 
	cadprolic_detalhe_fic (numlic, item, codigo, qtd, valor, qtdadt, valoradt, codccusto, qtdmed, valormed, tipo) 
	select numlic, item, item, quan1, vatomed1, quan1, vatomed1, codccusto, quan1, vatomed1, 'C' from cadprolic b where
	not exists (select 1 from cadprolic_detalhe_fic c where b.numlic = c.numlic and b.item = c.item);
""")
commit()

### PROPOSTA

In [ ]:
limpa_tabela(("cadpro", "cadpro_final", "cadpro_proposta"))

insert = cur_dest.prep("""
INSERT
    INTO
    cadpro_proposta(sessao,
    codif,
    item,
    itemp,
    quan1,
    vaun1,
    vato1,
    numlic,
    status,
    subem,
    marca,
    itemlance)
VALUES(?,?,?,?,?,?,?,?,?,?,?,?)""")

cur_orig.execute("""
with propostas as (
select
    codigo_da_pessoa::int codif,
    idp.numero_do_item::int item,
    pdp.quantidade_do_item_do_processo quan1,
    case
        when pdp.quantidade_do_item_do_processo <> 0 then pdp.valor_total / pdp.quantidade_do_item_do_processo
        else 0
    end vaun1,
    pdp.valor_total vato1,
    p.identificador_dos_processos_administrativos::int numlic,
    c.classificacao::int subem,
    c.proposta_vencedora,
    case
        when situacao_do_participante_item = 1 then 'C'
        else 'D'
    end status,
    pdp.marca_do_material marca,
    pdp.codigo_da_entidade::int empresa
from
    proposta_dos_participantes_itens pdp
join itens_do_processo_administrativo_gerados idp on
    pdp.identificador_do_item_do_processo_administrativo = idp.identificador_do_item_gerado
join participantes_da_licitacao p on
    pdp.identificador_do_participante = p.identificador_do_participante
join proposta_dos_participantes_classificacao c on
    c.identificador_da_proposta = pdp.identificador_da_proposta)
select distinct
    *
from
    propostas
--where empresa = 538
order by
    numlic,
    item,
    codif
""")           

for row in cur_orig:
    try:
        cur_dest.execute(insert, (1, row['codif'], row['item'], row['item'], row['quan1'], row['vaun1'], row['vato1'], row['numlic'], row['status'], row['subem'], to_cp1252_safe(row['marca']), 'S'))
    except Exception as e:
        print(f"Error inserting row: {e} - {row}")
        break
commit()

cur_dest.execute("""
MERGE INTO cadpro_proposta a USING (SELECT codif, codif_ant FROM desfor) b
ON a.codif = b.codif_ant
WHEN MATCHED THEN UPDATE SET a.codif = b.codif
""")
commit()

### CADPRO LANCE

In [ ]:
cur_dest.execute("""insert into cadpro_lance (sessao, rodada, codif, itemp, vaunl, vatol, status, subem, numlic)
	SELECT sessao, 1 rodada, CODIF, ITEMP, VAUN1, VATO1, 'F' status, SUBEM, numlic FROM CADPRO_PROPOSTA cp where subem = 1 and not exists
	(select 1 from cadpro_lance cl where cp.codif = cl.codif and cl.itemp = cp.itemp and cl.numlic = cp.numlic)""")
commit()

### CADPRO FINAL

In [ ]:
cur_dest.execute("""INSERT into cadpro_final (numlic, ult_sessao, codif, itemp, vaunf, vatof, STATUS, subem)
	SELECT numlic, sessao, codif, itemp, vaun1, vato1, CASE WHEN status = 'F' THEN 'C' ELSE status end, subem FROM cadpro_proposta
	WHERE NOT EXISTS (SELECT 1 FROM cadpro_final f WHERE f.numlic = cadpro_proposta.numlic AND f.itemp = cadpro_proposta.itemp AND f.codif = cadpro_proposta.codif)""")
commit()

cur_dest.execute("""update cadlic a SET comp = 3 WHERE comp = 1 AND EXISTS (SELECT 1 FROM cadpro_proposta x WHERE subem = 1 AND x.numlic = a.numlic) AND NOT EXISTS (SELECT 1 FROM cadpro x WHERE x.numlic = a.numlic)""")
commit()

### CADPRO_STATUS

In [ ]:
cur_dest.execute("""
    INSERT INTO cadpro_status (numlic, sessao, itemp, item, telafinal)
    SELECT b.NUMLIC, 1 AS sessao, a.item, a.item, 'I_ENCERRAMENTO'
    FROM CADPROLIC a
    JOIN cadlic b ON a.NUMLIC = b.NUMLIC
    WHERE NOT EXISTS (
        SELECT 1
        FROM cadpro_status c
        WHERE a.numlic = c.numlic
    )
    AND EXISTS (
        SELECT 1
        FROM cadlic d
        WHERE d.numlic = a.numlic AND d.comp = 3
    );
""")
commit()

### CADPRO

In [ ]:
limpa_tabela(("cadpro",))
cria_coluna("cadpro", "codant")

cur_dest.execute("""
INSERT INTO CADPRO (
  CODIF, CADPRO, QUAN1, VAUN1, VATO1, SUBEM, STATUS,
  ITEM, NUMORC, ITEMORCPED, CODCCUSTO, FICHA, ELEMENTO, DESDOBRO,
  NUMLIC, ULT_SESSAO, ITEMP, QTDADT, QTDPED, VAUNADT, VATOADT,
  PERC, QTDSOL, ID_CADORC, VATOPED, VATOSOL, TPCONTROLE_SALDO,
  QTDPED_FORNECEDOR_ANT, VATOPED_FORNECEDOR_ANT, MARCA, CODANT
)
WITH dados AS (
  SELECT
    a.CODIF,
      c.CADPRO,
      c.ITEM,
      d.NUMORC,
      c.CODCCUSTO,
      c.FICHA,
      c.ELEMENTO,
      c.DESDOBRO,
      a.NUMLIC,
      b.ITEMP,
      d.ID_CADORC,
      a.marca,
      a.quan1  AS qtdunit,
      a.VAUN1 AS vaun,
      a.vato1 AS vatotal,
      d.empresa
  FROM CADPRO_PROPOSTA a
  JOIN CADPRO_STATUS b
    ON b.NUMLIC = a.NUMLIC
    AND b.ITEMP  = a.ITEMP
    AND b.SESSAO = a.SESSAO
  JOIN CADPROLIC_DETALHE c
    ON c.NUMLIC = a.NUMLIC
    AND b.ITEMP  = c.ITEM_CADPROLIC
  JOIN CADLIC d
    ON d.NUMLIC = a.NUMLIC
  WHERE a.SUBEM = 1
    AND a.STATUS = 'C'
)
SELECT
    CODIF,
    CADPRO,
    qtdunit,
    vaun,
    vatotal,
    1,
    'C',
    ITEM,
    NUMORC,
    ITEM,
    CODCCUSTO,
    FICHA,
    ELEMENTO,
    DESDOBRO,
    NUMLIC,
    1,
    ITEMP,
    qtdunit,
    0,
    vaun,
    vatotal,
    0,
    0,
    ID_CADORC,
    0,
    0,
    'Q',
    0,
    0,
    marca,
    empresa
FROM dados d
WHERE NOT EXISTS (
    SELECT 1
    FROM CADPRO cp
    WHERE cp.NUMLIC = d.NUMLIC
      AND cp.ITEM   = d.ITEM
      AND cp.CODIF  = d.CODIF
)""")
commit()

### ADITIVOS

In [ ]:
cur_dest.execute("UPDATE CADPRO SET QTDADT = QUAN1, VAUNADT = VAUN1, VATOADT = VATO1 WHERE QTDADT <> QUAN1 OR VATOADT <> VATO1")
commit()

cur_orig.execute("""
with aditivos as (
select distinct
	cdpaec.identificador_dos_processos_administrativos_1::int numlic,
	idc.numero_do_item::int item,
	idc.quantidade_do_item_do_processo+(idc.quantidade_do_aditivo_de_acrescimo - idc.quantidade_do_aditivo_de_supressao ) qtdadt,
	novo_vl_unitario_perc vaunadt,
	c.entidade_de_contratacao::int entidade
from
	aditivos_itens ai
join itens_da_contratacao idc on
	ai.codigo_do_item_do_contrato = idc.identificador_do_item
join contratacao c on
	idc.contratacao = c.id_de_contratacao
join cache_dos_processos_administrativos_e_contratacoes cdpaec on
	c.identificador_do_cache_do_processo_administrativo = cdpaec.identificador_dos_processos_administrativos)
select *, qtdadt*vaunadt vatoadt from aditivos
--where entidade = 538
""")

update = cur_dest.prep("update cadpro set QTDADT = ?, VAUNADT = ?, VATOADT = ? where numlic = ? and item = ? and codant = ?")

for row in cur_orig:
	cur_dest.execute(update, (row['qtdadt'], row['vaunadt'], row['vatoadt'], row['numlic'], row['item'], row['entidade']))
	commit()

### REGPRECO

In [ ]:
limpa_tabela(("regprecodoc", "regpreco", "regprecohis"))

cur_dest.execute("""
EXECUTE BLOCK AS  
    BEGIN  
    INSERT INTO REGPRECODOC (NUMLIC, CODATUALIZACAO, DTPRAZO, ULTIMA)  
    SELECT DISTINCT A.NUMLIC, 0, DATEADD(1 YEAR TO A.DTHOM), 'S'  
    FROM CADLIC A WHERE A.REGISTROPRECO = 'S' AND A.DTHOM IS NOT NULL  
    AND NOT EXISTS(SELECT 1 FROM REGPRECODOC X  
    WHERE X.NUMLIC = A.NUMLIC);  

    INSERT INTO REGPRECO (COD, DTPRAZO, NUMLIC, CODIF, CADPRO, CODCCUSTO, ITEM, CODATUALIZACAO, QUAN1, VAUN1, VATO1, QTDENT, SUBEM, STATUS, ULTIMA)  
    SELECT B.ITEM, DATEADD(1 YEAR TO A.DTHOM), B.NUMLIC, B.CODIF, B.CADPRO, B.CODCCUSTO, B.ITEM, 0, B.QUAN1, B.VAUN1, B.VATO1, 0, B.SUBEM, B.STATUS, 'S'  
    FROM CADLIC A INNER JOIN CADPRO B ON (A.NUMLIC = B.NUMLIC) WHERE A.REGISTROPRECO = 'S' AND A.DTHOM IS NOT NULL  
    AND NOT EXISTS(SELECT 1 FROM REGPRECO X  
    WHERE X.NUMLIC = B.NUMLIC AND X.CODIF = B.CODIF AND X.CADPRO = B.CADPRO AND X.CODCCUSTO = B.CODCCUSTO AND X.ITEM = B.ITEM);  

    INSERT INTO REGPRECOHIS (NUMLIC, CODIF, CADPRO, CODCCUSTO, ITEM, CODATUALIZACAO, QUAN1, VAUN1, VATO1, SUBEM, STATUS, MOTIVO, MARCA, NUMORC, ULTIMA)  
    SELECT B.NUMLIC, B.CODIF, B.CADPRO, B.CODCCUSTO, B.ITEM, 0, B.QUAN1, B.VAUN1, B.VATO1, B.SUBEM, B.STATUS, B.MOTIVO, B.MARCA, B.NUMORC, 'S'  
    FROM CADLIC A INNER JOIN CADPRO B ON (A.NUMLIC = B.NUMLIC) WHERE A.REGISTROPRECO = 'S' AND A.DTHOM IS NOT NULL  
    AND NOT EXISTS(SELECT 1 FROM REGPRECOHIS X  
    WHERE X.NUMLIC = B.NUMLIC AND X.CODIF = B.CODIF AND X.CADPRO = B.CADPRO AND X.CODCCUSTO = B.CODCCUSTO AND X.ITEM = B.ITEM);  
END;""")
commit()

### ARQUIVOS

In [ ]:
limpa_tabela(("TRANSP_LICITACOES",))

## PEDIDOS

### ANTERIORES AO EXERCÍCIO

In [ ]:
limpa_tabela(("cadpro_saldo_ant", "regpreco_saldo_ant"))

insert = cur_dest.prep("""
insert into cadpro_saldo_ant (
    ano,
    numlic,
    item,
    cadpro,
    qtdped,
    vatoped)
values (?,?,?,?,?,?)
""")

cur_orig.execute("""
with cte as
(select
    to_char(s.numero_da_solicitacao_de_fornecimento , 'fm00000/')||ano%2000 numped,
    to_char(s.numero_da_solicitacao_de_fornecimento , 'fm00000') num,
    ano::int,
    s.data_da_solicitacao::date datped,
    s.identificador_do_fornecedor::int codif,
    s.codigo_do_item_do_apostilamento::int id_cadped,
    s.codigo_da_entidade::int empresa,
    c.identificador_dos_processos_administrativos_1::int numlic,
    s.observacao obs,
    numero_que_representa_o_organograma codccusto,
    coalesce(ic.numero_do_item, i.numero_do_item_da_solicitacao_de_fornecimento)::int item,
    i.codigo_do_material::varchar codreduz,
    i.quantidade_do_item qtd,
    i.valor_unitario prcunt,
    i.valor_total prctot,
    null id_cadorc,
    to_char(con.numero_termo_de_contratacao, 'fm0000/')||con.ano_termo_de_contratacao%2000 contrato
from
    solicitacoes_de_fornecimento s
join cache_dos_processos_administrativos_e_contratacoes c on c.identificador_dos_processos_administrativos = s.i_processos_administrativos
left join organogramas_contratos o on o.codigo_dos_organogramas_contratos = s.codigo_do_organograma 
left join itens_da_solicitacao_de_fornecimento i on i.codigo_da_solicitacao_de_fornecimento = s.codigo_do_item_do_apostilamento
left join itens_da_contratacao ic on ic.identificador_do_item = i.codigo_do_item_da_contratacao 
left join contratacao con on s.codigo_da_contratacao = con.id_de_contratacao
where ano < 2025 
--and s.codigo_da_entidade = 538 
and i.codigo_do_material is not null)
select numlic, codif, item, codreduz, sum(qtd) qtdped, sum(prctot) vatoped from cte group by 1,2,3,4
order by 1,2
""")

ano_ant = int(exercicio)-1

for row in cur_orig:
    cadpro = cadest[row['codreduz']]

    try:
        cur_dest.execute(insert, (
            ano_ant,
            row['numlic'],
            row['item'],
            cadpro,
            row['qtdped'],
            row['vatoped']
        ))
        commit()
    except fdb.DatabaseError as e:
        if e.args[1] == -803: 
            cur_dest.execute("update cadpro_saldo_ant set qtdped = qtdped + ?, vatoped = vatoped + ? where numlic = ? and item = ?", (row['qtdped'], row['vatoped'], row['numlic'], row['item'])) 
            commit()
        else:
            raise

cur_dest.execute("""
insert into regpreco_saldo_ant (ano, item, cadpro, qtdent, numlic, vatoent) select ano, item, cadpro, qtdped, numlic, vatoped from cadpro_saldo_ant 
where exists (select 1 from cadlic where cadpro_saldo_ant.numlic = cadlic.numlic and cadlic.registropreco = 'S')
""")
commit()

### DO EXERCÍCIO

In [10]:
limpa_tabela(("icadped", "cadped"))

inser_cadped = cur_dest.prep("insert into cadped (numped, num, ano, datped, codif, id_cadped, empresa, numlic, obs, codccusto, id_cadorc, contrato) values (?,?,?,?,?,?,?,?,?,?,?,?)")
insert_icadped = cur_dest.prep("insert into icadped (numped, item, cadpro, qtd, prcunt, prctot, codccusto, id_cadped, qtdanu, prctotanu) values (?,?,?,?,?,?,?,?,?,?)")

cur_orig.execute(f"""
with ANULACOES as (select
	codigo_do_item_solicitacao_de_fornecimento,
	sfai.quantidade_anulada,
	sfai.valor_anulado 
from
	sol_fornec_anulac_item sfai),
CENTROCUSTO as (with centrocusto as (
select
    oc.numero_que_representa_o_organograma codant,
    max(oc.descricao) descr,
    max(codigo_dos_organogramas_contratos) id
from
    organogramas_contratos oc
group by
    1)
select
    substr(codant, 1, 2) poder,
    substr(codant, 4, 2) orgao,
    --substr(codant,9,2) unidade,
    dense_rank() over (order by id) codccusto,
    codant,
    descr,
    '000000001' destino
from
    centrocusto)
select
    to_char(s.numero_da_solicitacao_de_fornecimento , 'fm00000/')||ano%2000 numped,
    to_char(s.numero_da_solicitacao_de_fornecimento , 'fm00000') num,
    ano::int,
    s.data_da_solicitacao::date datped,
    s.identificador_do_fornecedor::int codif,
    s.codigo_do_item_do_apostilamento::int id_cadped,
    s.codigo_da_entidade::int empresa,
    c.identificador_dos_processos_administrativos_1::int numlic,
    s.observacao obs,
    concat(s.codigo_da_entidade, codccusto)::int codccusto,
    coalesce(ic.numero_do_item, i.numero_do_item_da_solicitacao_de_fornecimento)::int item,
    i.codigo_do_material::varchar codreduz,
    i.quantidade_do_item qtd,
    i.valor_unitario prcunt,
    i.valor_total prctot,
    null id_cadorc,
    to_char(con.numero_termo_de_contratacao, 'fm0000/')||con.ano_termo_de_contratacao%2000 contrato,
    quantidade_anulada,
    valor_anulado
from
    solicitacoes_de_fornecimento s
join cache_dos_processos_administrativos_e_contratacoes c on c.identificador_dos_processos_administrativos = s.i_processos_administrativos
left join organogramas_contratos o on o.codigo_dos_organogramas_contratos = s.codigo_do_organograma 
left join itens_da_solicitacao_de_fornecimento i on i.codigo_da_solicitacao_de_fornecimento = s.codigo_do_item_do_apostilamento
left join itens_da_contratacao ic on ic.identificador_do_item = i.codigo_do_item_da_contratacao 
left join contratacao con on s.codigo_da_contratacao = con.id_de_contratacao
left join CENTROCUSTO on CENTROCUSTO.codant = numero_que_representa_o_organograma
left join ANULACOES a on a.codigo_do_item_solicitacao_de_fornecimento = i.codigo_do_item_do_apostilamento
where ano = {exercicio}
--and s.codigo_da_entidade = 538
and i.codigo_do_material is not null
union all
select
    to_char(s.numero_da_solicitacao_de_fornecimento , 'fm00000/')||ano%2000 numped,
    to_char(s.numero_da_solicitacao_de_fornecimento , 'fm00000') num,
    ano::int,
    s.data_da_solicitacao::date datped,
    s.identificador_do_fornecedor::int codif,
    s.codigo_do_item_do_apostilamento::int id_cadped,
    s.codigo_da_entidade::int empresa,
    null numlic,
    s.observacao obs,
    concat(s.codigo_da_entidade, codccusto)::int codccusto,
    coalesce(ic.numero_do_item, i.numero_do_item_da_solicitacao_de_fornecimento)::int item,
    i.codigo_do_material::varchar codreduz,
    i.quantidade_do_item qtd,
    i.valor_unitario prcunt,
    i.valor_total prctot,
    c.id_solicitacao_compra::int id_cadorc,
    to_char(c.numero_termo_de_contratacao, 'fm0000/')||c.ano_termo_de_contratacao%2000 contrato,
    quantidade_anulada,
    valor_anulado
from
    solicitacoes_de_fornecimento s
left join organogramas_contratos o on o.codigo_dos_organogramas_contratos = s.codigo_do_organograma 
join itens_da_solicitacao_de_fornecimento i on i.codigo_da_solicitacao_de_fornecimento = s.codigo_do_item_do_apostilamento
join itens_da_contratacao ic on ic.identificador_do_item = i.codigo_do_item_da_contratacao 
join contratacao c on s.codigo_da_contratacao = c.id_de_contratacao and identificador_do_cache_do_processo_administrativo is null and id_solicitacao_compra is not null
left join CENTROCUSTO on CENTROCUSTO.codant = numero_que_representa_o_organograma
left join ANULACOES a on a.codigo_do_item_solicitacao_de_fornecimento = i.codigo_do_item_do_apostilamento
where i_processos_administrativos is null
and ano = {exercicio}
--and s.codigo_da_entidade = 538
and i.codigo_do_material is not null
union all
select
    to_char(s.numero_da_solicitacao_de_fornecimento , 'fm00000/')||ano%2000 numped,
    to_char(s.numero_da_solicitacao_de_fornecimento , 'fm00000') num,
    ano::int,
    s.data_da_solicitacao::date datped,
    s.identificador_do_fornecedor::int codif,
    s.codigo_do_item_do_apostilamento::int id_cadped,
    s.codigo_da_entidade::int empresa,
    null numlic,
    s.observacao obs,
    concat(s.codigo_da_entidade, codccusto)::int codccusto,
    coalesce(ic.numero_do_item, i.numero_do_item_da_solicitacao_de_fornecimento)::int item,
    i.codigo_do_material::varchar codreduz,
    i.quantidade_do_item qtd,
    i.valor_unitario prcunt,
    i.valor_total prctot,
    c.id_solicitacao_compra::int id_cadorc,
    to_char(c.numero_termo_de_contratacao, 'fm0000/')||c.ano_termo_de_contratacao%2000 contrato,
    quantidade_anulada,
    valor_anulado
from
    solicitacoes_de_fornecimento s
left join organogramas_contratos o on o.codigo_dos_organogramas_contratos = s.codigo_do_organograma 
join itens_da_solicitacao_de_fornecimento i on i.codigo_da_solicitacao_de_fornecimento = s.codigo_do_item_do_apostilamento
join itens_da_contratacao ic on ic.identificador_do_item = i.codigo_do_item_da_contratacao 
join contratacao c on s.codigo_da_contratacao = c.id_de_contratacao and identificador_do_cache_do_processo_administrativo is null and id_solicitacao_compra is null
left join CENTROCUSTO on CENTROCUSTO.codant = numero_que_representa_o_organograma
left join ANULACOES a on a.codigo_do_item_solicitacao_de_fornecimento = i.codigo_do_item_do_apostilamento
where i_processos_administrativos is null 
and ano = {exercicio}
--and s.codigo_da_entidade = 538 
and i.codigo_do_material is not null
order by num, empresa, item
""")

id_cadped = 0

for row in cur_orig:
    try:
        if row['id_cadped'] != id_cadped:
            id_cadped = row['id_cadped']
            cur_dest.execute(inser_cadped, (
                row['numped'],
                row['num'],
                row['ano'],
                row['datped'],
                row['codif'],
                row['id_cadped'],
                row['empresa'],
                row['numlic'],
                row['obs'],
                row['codccusto'],
                row['id_cadorc'],
                row['contrato']
            ))
            commit()
    except Exception as e:
        print(f'Erro ao inserir pedido {row["num"]}: {e} - {row}')
        break
    
    cadpro = cadest[row['codreduz']]

    try:
        cur_dest.execute(insert_icadped, (
            row['numped'],
            row['item'],
            cadpro,
            row['qtd'],
            row['prcunt'],
            row['prctot'],
            row['codccusto'],
            id_cadped,
            row['quantidade_anulada'],
            row['valor_anulado']
        ))
    except Exception as e:
        print(f'Erro ao inserir item pedido {row['num']}: {e} - {row}')
        break
commit()

cur_dest.execute("""
UPDATE cadped a SET a.codatualizacao_rp = (SELECT max(codatualizacao) 
FROM regprecodoc x WHERE x.numlic = a.numlic)
""")
commit()

cur_dest.execute("""
MERGE INTO cadped a USING (SELECT codif, codif_ant FROM desfor) b
ON a.codif = b.codif_ant
WHEN MATCHED THEN UPDATE SET a.codif = b.codif
""")
commit()

cur_dest.execute("""
EXECUTE BLOCK
AS
DECLARE VARIABLE NUMLIC INTEGER;
DECLARE VARIABLE ITEM INTEGER;
DECLARE VARIABLE CODCCUSTO INTEGER;
BEGIN
  FOR
      WITH freq AS (
          SELECT
              numlic,
              a.item,
              a.codccusto,
              COUNT(*) AS qtd,
              ROW_NUMBER() OVER (
                  PARTITION BY numlic, a.item
                  ORDER BY COUNT(*) DESC, a.codccusto
              ) AS rn
          FROM icadped a
          JOIN cadped c USING (id_cadped)
          GROUP BY numlic, a.item, a.codccusto
      )
      SELECT
          numlic,
          item,
          codccusto
      FROM freq
      WHERE rn = 1
      INTO :NUMLIC, :ITEM, :CODCCUSTO
  DO BEGIN
      UPDATE CADPROLIC p
         SET p.CODCCUSTO = :CODCCUSTO
       WHERE p.ITEM = :ITEM
         AND p.NUMLIC = :NUMLIC;

      UPDATE CADPROLIC_DETALHE d
         SET d.CODCCUSTO = :CODCCUSTO
       WHERE d.ITEM = :ITEM
         AND d.NUMLIC = :NUMLIC;

      UPDATE CADPROLIC_DETALHE_FIC f
         SET f.CODCCUSTO = :CODCCUSTO
       WHERE f.ITEM = :ITEM
         AND f.NUMLIC = :NUMLIC;

      UPDATE CADPRO c
         SET c.CODCCUSTO = :CODCCUSTO
       WHERE c.ITEM = :ITEM
         AND c.NUMLIC = :NUMLIC;

      UPDATE REGPRECO r
         SET r.CODCCUSTO = :CODCCUSTO
       WHERE r.ITEM = :ITEM
         AND r.NUMLIC = :NUMLIC;

      UPDATE REGPRECOHIS h
         SET h.CODCCUSTO = :CODCCUSTO
       WHERE h.ITEM = :ITEM
         AND h.NUMLIC = :NUMLIC;
  END
END
""")
commit()

# 📄 CONTRATOS

## CADASTRO

In [ ]:
limpa_tabela(['contratos'])
cria_coluna('contratos', 'codant')

cur_orig.execute(f"""
with cadlic as (with cadorc as (select
	identificador_dos_processos_administrativos numlic,
	max(s.codigo_dos_organograma) codigo_dos_organograma,
	max(coalesce(cp.identificador_de_registro,s.identificador_das_solicitacoes)) id_cadorc,
	max(concat(to_char(coalesce(numero_da_cotacao::text,substr(s.codigo_sequencial_das_solicitacoes::text,1,5))::int,'fm00000/'),coalesce(exc.exercicio, ex.exercicio)%2000)) numorc,
	case when cp.identificador_de_registro is not null then 'C' else 'S' end tipo_origem
from
	processo_de_compra_x_solicitacao pxs
join solicitacoes s on pxs.identificador_das_solicitacoes = s.identificador_das_solicitacoes 
join parametros_por_exercicio ex on ex.codigo_sequencial = s.codigo_sequencial 
left join solicitacoes_x_cotacoes sxc on s.identificador_das_solicitacoes = sxc.identificador_das_solicitacoes 
left join cotacoes_de_preco cp on cp.identificador_de_registro = sxc.identificador_de_registro 
left join parametros_por_exercicio exc on exc.codigo_sequencial = cp.codigo_sequencial
group by 1,5)
select
	pa.identificador_dos_processos_administrativos::int numlic,
	concat(to_char(pa.codigo_sequencial_das_solicitacoes, 'fm000000/'),exercicio%2000) proclic,
	pa.codigo_sequencial_das_solicitacoes::int numero,
	exercicio::int ano,
	case when pa.status_do_processo = 11 then 3 when pa.status_do_processo = 0 then 1 else 2 end comp,
	1 licnova,
	case when pa.status_do_processo = 11 then 'S' else 'N' end liberacompra,
	pa.objeto_do_processo discr,
	tdo.descricao detalhe,
	registro_de_preco registropreco,
	3 microempresa,
	numeracao_forma_contratacao::int numpro,
	/*fdj.descricao*/ 'Menor Preco Unitario' discr7,
	pa.data_do_processo::date datae,
	data_do_processo::date processo_data,
	paaf.data_e_hora_do_ato::date dtajd,
	paaf.data_e_hora_do_ato::date dthom,
	null codtce,
	c.ano_sequencial::int anomod,
	mdl.sigla modlic,
	null licit,
	null codmod,
	pdp.data_da_publicacao::date dtpub,
	paaf.data_e_hora_do_ato::date dtenc,
	pa.valor_total_dos_itens valor,
	pa.codigo_da_entidade::int empresa,
	numero_protocolo processo,
	pa.ano_do_protocolo processo_ano,
	data_hora_abertura_da_sessao_julgamento::date dtreal,
	data_e_hora_abertura_envelopes::date dtenv,
	numorc,
	id_cadorc::int,
	case when pa.controle_de_saldo = 'V' then 'T' else controle_de_saldo end tpcontrole_saldo,
	data_e_hora_inicio_recebimento_envelopes::date dtpropostaini,
	data_e_hora_final_recebimento_envelopes::date dtpropostafim,
	coalesce(tipo_origem, 'D') tipo_origem
from
	processos_administrativos pa
join parametros_por_exercicio pe on pe.codigo_sequencial = pa.codigo_sequencial
left join formas_de_julgamento fdj on fdj.identificador_do_tipo_do_objeto = pa.identificador_da_forma_julgamento 
left join tipos_de_objetos tdo on tdo.identificador_do_tipo_do_objeto = pa.identificador_do_tipo_do_objeto
left join (select identificador_dos_processos_administrativos, max(data_da_publicacao) data_da_publicacao from publicacoes_do_processo group by 1) pdp on pdp.identificador_dos_processos_administrativos = pa.identificador_dos_processos_administrativos 
left join cadorc on cadorc.numlic = pa.identificador_dos_processos_administrativos
left join contratacoes c on c.identificador_dos_processos_administrativos = pa.identificador_dos_processos_administrativos
left join modalidades_de_licitacao mdl on c.identificador_de_registro = mdl.identificador_de_registro
left join processos_administrativos_atos_finais paaf on paaf.identificador_dos_processos_administrativos = pa.identificador_dos_processos_administrativos and tipo_do_ato = 3
left join (select row_number() over (partition by identificador_dos_processos_administrativos_contratacoes order by identificador_das_sessoes_de_julgamento) seq, * from processos_administrativos_sessoes_de_julgamento) pasdj 
on pasdj.identificador_dos_processos_administrativos_contratacoes = pa.identificador_dos_processos_administrativos and seq = 1
--where pa.codigo_da_entidade = 538
and numeracao_forma_contratacao is not null)
select
	to_char(row_number() over (partition by ano_termo_de_contratacao), 'fm0000/')||ano_termo_de_contratacao%2000 codigo,
	to_char(c.codigo_sequencial_do_contrato, 'fm0000/')||ano_termo_de_contratacao%2000 contratonum,
	c.entidade_de_contratacao::bigint empresa,
	ano_termo_de_contratacao::bigint ano,
	c.data_da_assinatura_do_contrato::date dtassi,
	c.data_de_inicio_da_vigencia_do_contrato::date vigeni,
	c.data_final_da_vigencia_do_contrato::date vigenf,
	c.data_da_assinatura_do_contrato::date dtpubl,
	cadlic.numero::bigint nproli,
	'LICITAÇÃO' flegal,
	c.codigo_do_fornecedor::int codif_ant,
	c.objeto_da_contratacao objeto,
	coalesce(c.valor_original_do_contrato,0) valcon,
	'P' tipo_contrato,
	case when c.identificador_da_ata_de_registro_de_preco is not null then 'S' else 'N' end registropreco,
	identificador_do_tipo_de_objeto tipoco,
	proclic,
	numero_termo_de_contratacao::bigint codant,
	numlic
from
	contratacao c
left join cache_dos_processos_administrativos_e_contratacoes cp on cp.identificador_dos_processos_administrativos = c.identificador_do_cache_do_processo_administrativo
left join cadlic on cp.identificador_dos_processos_administrativos_1 = numlic
where to_char(c.codigo_sequencial_do_contrato, 'fm0000/')||ano_termo_de_contratacao%2000 is not null
""")

insert = cur_dest.prep("""
INSERT
	INTO
	contratos (codigo,
	contratonum,
	empresa,
	ano,
	dtassi,
	vigeni,
	vigenf,
	dtpubl,
	nproli,
	flegal,
	fundlegal,
	codif,
	objeti,
	objeto,
	objeto_completo,
	valcon,
	tipo_contrato,
	ataregpreco,
	tipoco,
	proclic,
	codant,
	numlic) values (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

tipoco = {
    1614: '04', # Locação
    3680: '01', # Compra
    1615: '02',
    132: '02',
    10: '13'
}

for row in cur_orig:
    try:
        cur_dest.execute(insert, (
            row['codigo'],
            row['contratonum'],
            row['empresa'],
            row['ano'],
            row['dtassi'],
            row['vigeni'],
            row['vigenf'],
            row['dtpubl'],
            row['nproli'],
            row['flegal'],
            row['flegal'],
            row['codif_ant'],
            row['objeto'][:60] if row['objeto'] else None,
            row['objeto'][:60] if row['objeto'] else None,
            to_cp1252_safe(row['objeto']) if row['objeto'] else None,
            row['valcon'],
            row['tipo_contrato'],
            row['registropreco'],
            tipoco.get(row['tipoco'], '99'),
            row['proclic'],
            row['codant'],
            row['numlic']
        ))
    except Exception as e:
        print(f'Erro ao inserir contrato {row["codigo"]}: {e} - {row}')
        break
    
cur_dest.execute("""
MERGE INTO contratos a USING (SELECT codif, codif_ant FROM desfor) b
ON a.codif = b.codif_ant
WHEN MATCHED THEN UPDATE SET a.codif = b.codif
""")
commit()

## ITENS

In [ ]:
limpa_tabela(["CONTRATOSITEMLICIT"])

# cur_dest.execute("""
# INSERT INTO CONTRATOSITEMLICIT (CONTRATO, ITEM, LOTELIC, CADPRO, QTD, UND, DESCR, VLUNIT, VLTOTAL) 
# SELECT
# 	d.codigo,
# 	ITEM,
#     '',
# 	CADPRO,
# 	a.QUAN1,
# 	substring(c.UNID1 from 1 for 3),
# 	c.DISC1,
# 	a.VAUN1,
# 	a.VATO1
# FROM
# 	CADPRO a
# JOIN cadlic b USING (numlic)
# JOIN cadest c USING (cadpro)
# JOIN contratos d USING (numlic)
# WHERE NOT EXISTS (SELECT 1 FROM CONTRATOSITEMLICIT x WHERE x.contrato = d.codigo AND x.item = a.item) and a.subem = 1
# """)
# commit()

cur_orig.execute("""
with CONTRATOS as (
    with cadlic as (with cadorc as (select
        identificador_dos_processos_administrativos numlic,
        max(s.codigo_dos_organograma) codigo_dos_organograma,
        max(coalesce(cp.identificador_de_registro,s.identificador_das_solicitacoes)) id_cadorc,
        max(concat(to_char(coalesce(numero_da_cotacao::text,substr(s.codigo_sequencial_das_solicitacoes::text,1,5))::int,'fm00000/'),coalesce(exc.exercicio, ex.exercicio)%2000)) numorc,
        case when cp.identificador_de_registro is not null then 'C' else 'S' end tipo_origem
    from
        processo_de_compra_x_solicitacao pxs
    join solicitacoes s on pxs.identificador_das_solicitacoes = s.identificador_das_solicitacoes 
    join parametros_por_exercicio ex on ex.codigo_sequencial = s.codigo_sequencial 
    left join solicitacoes_x_cotacoes sxc on s.identificador_das_solicitacoes = sxc.identificador_das_solicitacoes 
    left join cotacoes_de_preco cp on cp.identificador_de_registro = sxc.identificador_de_registro 
    left join parametros_por_exercicio exc on exc.codigo_sequencial = cp.codigo_sequencial
    group by 1,5)
    select
        pa.identificador_dos_processos_administrativos::int numlic,
        concat(to_char(pa.codigo_sequencial_das_solicitacoes, 'fm000000/'),exercicio%2000) proclic,
        pa.codigo_sequencial_das_solicitacoes::int numero,
        exercicio::int ano,
        case when pa.status_do_processo = 11 then 3 when pa.status_do_processo = 0 then 1 else 2 end comp,
        1 licnova,
        case when pa.status_do_processo = 11 then 'S' else 'N' end liberacompra,
        pa.objeto_do_processo discr,
        tdo.descricao detalhe,
        registro_de_preco registropreco,
        3 microempresa,
        numeracao_forma_contratacao::int numpro,
        /*fdj.descricao*/ 'Menor Preco Unitario' discr7,
        pa.data_do_processo::date datae,
        data_do_processo::date processo_data,
        paaf.data_e_hora_do_ato::date dtajd,
        paaf.data_e_hora_do_ato::date dthom,
        null codtce,
        c.ano_sequencial::int anomod,
        mdl.sigla modlic,
        null licit,
        null codmod,
        pdp.data_da_publicacao::date dtpub,
        paaf.data_e_hora_do_ato::date dtenc,
        pa.valor_total_dos_itens valor,
        pa.codigo_da_entidade::int empresa,
        numero_protocolo processo,
        pa.ano_do_protocolo processo_ano,
        data_hora_abertura_da_sessao_julgamento::date dtreal,
        data_e_hora_abertura_envelopes::date dtenv,
        numorc,
        id_cadorc::int,
        case when pa.controle_de_saldo = 'V' then 'T' else controle_de_saldo end tpcontrole_saldo,
        data_e_hora_inicio_recebimento_envelopes::date dtpropostaini,
        data_e_hora_final_recebimento_envelopes::date dtpropostafim,
        coalesce(tipo_origem, 'D') tipo_origem
    from
        processos_administrativos pa
    join parametros_por_exercicio pe on pe.codigo_sequencial = pa.codigo_sequencial
    left join formas_de_julgamento fdj on fdj.identificador_do_tipo_do_objeto = pa.identificador_da_forma_julgamento 
    left join tipos_de_objetos tdo on tdo.identificador_do_tipo_do_objeto = pa.identificador_do_tipo_do_objeto
    left join (select identificador_dos_processos_administrativos, max(data_da_publicacao) data_da_publicacao from publicacoes_do_processo group by 1) pdp on pdp.identificador_dos_processos_administrativos = pa.identificador_dos_processos_administrativos 
    left join cadorc on cadorc.numlic = pa.identificador_dos_processos_administrativos
    left join contratacoes c on c.identificador_dos_processos_administrativos = pa.identificador_dos_processos_administrativos
    left join modalidades_de_licitacao mdl on c.identificador_de_registro = mdl.identificador_de_registro
    left join processos_administrativos_atos_finais paaf on paaf.identificador_dos_processos_administrativos = pa.identificador_dos_processos_administrativos and tipo_do_ato = 3
    left join (select row_number() over (partition by identificador_dos_processos_administrativos_contratacoes order by identificador_das_sessoes_de_julgamento) seq, * from processos_administrativos_sessoes_de_julgamento) pasdj 
    on pasdj.identificador_dos_processos_administrativos_contratacoes = pa.identificador_dos_processos_administrativos and seq = 1
    --where pa.codigo_da_entidade = 538
    and numeracao_forma_contratacao is not null)
    select
        to_char(row_number() over (partition by ano_termo_de_contratacao), 'fm0000/')||ano_termo_de_contratacao%2000 codigo,
        to_char(c.codigo_sequencial_do_contrato, 'fm0000/')||ano_termo_de_contratacao%2000 contratonum,
        c.entidade_de_contratacao::bigint empresa,
        ano_termo_de_contratacao::bigint ano,
        c.data_da_assinatura_do_contrato::date dtassi,
        c.data_de_inicio_da_vigencia_do_contrato::date vigeni,
        c.data_final_da_vigencia_do_contrato::date vigenf,
        c.data_da_assinatura_do_contrato::date dtpubl,
        cadlic.numero::bigint nproli,
        'LICITAÇÃO' flegal,
        c.codigo_do_fornecedor::int codif_ant,
        c.objeto_da_contratacao objeto,
        coalesce(c.valor_original_do_contrato,0) valcon,
        'P' tipo_contrato,
        case when c.identificador_da_ata_de_registro_de_preco is not null then 'S' else 'N' end registropreco,
        identificador_do_tipo_de_objeto tipoco,
        proclic,
        numero_termo_de_contratacao::bigint codant,
        numlic,
        c.id_de_contratacao
    from
        contratacao c
    left join cache_dos_processos_administrativos_e_contratacoes cp on cp.identificador_dos_processos_administrativos = c.identificador_do_cache_do_processo_administrativo
    left join cadlic on cp.identificador_dos_processos_administrativos_1 = numlic
    where to_char(c.codigo_sequencial_do_contrato, 'fm0000/')||ano_termo_de_contratacao%2000 is not null
)
select 
    codigo,
    numero_do_item::int item,
    referencia_material codreduz,
    sum(quantidade_do_item_do_processo) qtd,
    valor_unitario vlunit,
    sum(valor_total) vltotal
from itens_da_contratacao idc 
join CONTRATOS c on c.id_de_contratacao = idc.contratacao 
group by 1,2,3,5""")

insert = cur_dest.prep("""
INSERT INTO CONTRATOSITEMLICIT (CONTRATO, ITEM, LOTELIC, CADPRO, QTD, UND, DESCR, VLUNIT, VLTOTAL) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

for row in cur_orig:
    cadpro = cadest[str(row['codreduz'])]
    if not cadpro:
        print(f'Produto {row["codreduz"]} do contrato {row["codigo"]} não encontrado no cadastro de estoque.')
        continue

    try:
        cur_dest.execute(insert, (
            row['codigo'],
            row['item'],
            '',
            cadpro,
            row['qtd'],
            '', 
            '', 
            row['vlunit'],
            row['vltotal']
        ))
    except Exception as e:
        print(f'Erro ao inserir item contrato {row["codigo"]}: {e} - {row}')
        break
commit()

## ADITIVOS

In [ ]:
limpa_tabela(["CONTRATOSADITAMENTO"])

cur_orig.execute(f"""
with CONTRATOS as (
	with cadlic as (with cadorc as (select
		identificador_dos_processos_administrativos numlic,
		max(s.codigo_dos_organograma) codigo_dos_organograma,
		max(coalesce(cp.identificador_de_registro,s.identificador_das_solicitacoes)) id_cadorc,
		max(concat(to_char(coalesce(numero_da_cotacao::text,substr(s.codigo_sequencial_das_solicitacoes::text,1,5))::int,'fm00000/'),coalesce(exc.exercicio, ex.exercicio)%2000)) numorc,
		case when cp.identificador_de_registro is not null then 'C' else 'S' end tipo_origem
	from
		processo_de_compra_x_solicitacao pxs
	join solicitacoes s on pxs.identificador_das_solicitacoes = s.identificador_das_solicitacoes 
	join parametros_por_exercicio ex on ex.codigo_sequencial = s.codigo_sequencial 
	left join solicitacoes_x_cotacoes sxc on s.identificador_das_solicitacoes = sxc.identificador_das_solicitacoes 
	left join cotacoes_de_preco cp on cp.identificador_de_registro = sxc.identificador_de_registro 
	left join parametros_por_exercicio exc on exc.codigo_sequencial = cp.codigo_sequencial
	group by 1,5)
	select
		pa.identificador_dos_processos_administrativos::int numlic,
		concat(to_char(pa.codigo_sequencial_das_solicitacoes, 'fm000000/'),exercicio%2000) proclic,
		pa.codigo_sequencial_das_solicitacoes::int numero,
		exercicio::int ano,
		case when pa.status_do_processo = 11 then 3 when pa.status_do_processo = 0 then 1 else 2 end comp,
		1 licnova,
		case when pa.status_do_processo = 11 then 'S' else 'N' end liberacompra,
		pa.objeto_do_processo discr,
		tdo.descricao detalhe,
		registro_de_preco registropreco,
		3 microempresa,
		numeracao_forma_contratacao::int numpro,
		/*fdj.descricao*/ 'Menor Preco Unitario' discr7,
		pa.data_do_processo::date datae,
		data_do_processo::date processo_data,
		paaf.data_e_hora_do_ato::date dtajd,
		paaf.data_e_hora_do_ato::date dthom,
		null codtce,
		c.ano_sequencial::int anomod,
		mdl.sigla modlic,
		null licit,
		null codmod,
		pdp.data_da_publicacao::date dtpub,
		paaf.data_e_hora_do_ato::date dtenc,
		pa.valor_total_dos_itens valor,
		pa.codigo_da_entidade::int empresa,
		numero_protocolo processo,
		pa.ano_do_protocolo processo_ano,
		data_hora_abertura_da_sessao_julgamento::date dtreal,
		data_e_hora_abertura_envelopes::date dtenv,
		numorc,
		id_cadorc::int,
		case when pa.controle_de_saldo = 'V' then 'T' else controle_de_saldo end tpcontrole_saldo,
		data_e_hora_inicio_recebimento_envelopes::date dtpropostaini,
		data_e_hora_final_recebimento_envelopes::date dtpropostafim,
		coalesce(tipo_origem, 'D') tipo_origem
	from
		processos_administrativos pa
	join parametros_por_exercicio pe on pe.codigo_sequencial = pa.codigo_sequencial
	left join formas_de_julgamento fdj on fdj.identificador_do_tipo_do_objeto = pa.identificador_da_forma_julgamento 
	left join tipos_de_objetos tdo on tdo.identificador_do_tipo_do_objeto = pa.identificador_do_tipo_do_objeto
	left join (select identificador_dos_processos_administrativos, max(data_da_publicacao) data_da_publicacao from publicacoes_do_processo group by 1) pdp on pdp.identificador_dos_processos_administrativos = pa.identificador_dos_processos_administrativos 
	left join cadorc on cadorc.numlic = pa.identificador_dos_processos_administrativos
	left join contratacoes c on c.identificador_dos_processos_administrativos = pa.identificador_dos_processos_administrativos
	left join modalidades_de_licitacao mdl on c.identificador_de_registro = mdl.identificador_de_registro
	left join processos_administrativos_atos_finais paaf on paaf.identificador_dos_processos_administrativos = pa.identificador_dos_processos_administrativos and tipo_do_ato = 3
	left join (select row_number() over (partition by identificador_dos_processos_administrativos_contratacoes order by identificador_das_sessoes_de_julgamento) seq, * from processos_administrativos_sessoes_de_julgamento) pasdj 
	on pasdj.identificador_dos_processos_administrativos_contratacoes = pa.identificador_dos_processos_administrativos and seq = 1
	--where pa.codigo_da_entidade = 538
	and numeracao_forma_contratacao is not null)
	select
		to_char(row_number() over (partition by ano_termo_de_contratacao), 'fm0000/')||ano_termo_de_contratacao%2000 codigo,
		to_char(c.codigo_sequencial_do_contrato, 'fm0000/')||ano_termo_de_contratacao%2000 contratonum,
		c.entidade_de_contratacao::bigint empresa,
		ano_termo_de_contratacao::bigint ano,
		c.data_da_assinatura_do_contrato::date dtassi,
		c.data_de_inicio_da_vigencia_do_contrato::date vigeni,
		c.data_final_da_vigencia_do_contrato::date vigenf,
		c.data_da_assinatura_do_contrato::date dtpubl,
		cadlic.numero::bigint nproli,
		'LICITAÇÃO' flegal,
		c.codigo_do_fornecedor::int codif_ant,
		c.objeto_da_contratacao objeto,
		coalesce(c.valor_original_do_contrato,0) valcon,
		'P' tipo_contrato,
		case when c.identificador_da_ata_de_registro_de_preco is not null then 'S' else 'N' end registropreco,
		identificador_do_tipo_de_objeto tipoco,
		proclic,
		numero_termo_de_contratacao::bigint codant,
		numlic,
		c.id_de_contratacao
	from
		contratacao c
	left join cache_dos_processos_administrativos_e_contratacoes cp on cp.identificador_dos_processos_administrativos = c.identificador_do_cache_do_processo_administrativo
	left join cadlic on cp.identificador_dos_processos_administrativos_1 = numlic
	where to_char(c.codigo_sequencial_do_contrato, 'fm0000/')||ano_termo_de_contratacao%2000 is not null
)
select
	codigo,
	data_de_assinatura_do_aditivo::date dtlan,
	nova_data_de_finalizacao::date dataencerramento,
	valor_do_aditivo valor,
	objeto_do_aditivo descricao,
	data_e_hora_da_criacao::date datainsc,
	case when codigo_do_tipo_de_aditivo in (3,6,7,9,10) then 'Acréscimo' when codigo_do_tipo_de_aditivo in (13) then 'Prorrogação' else 'Decréscimo' end TIPOHIST,
	'Bilateral' TIPOALT,
	to_char(numero_sequencial_do_aditivo, 'fm00000/')||extract(year from data_de_assinatura_do_aditivo)%2000 termo
from
	aditivos_da_contratacao adc
join CONTRATOS c on c.id_de_contratacao = adc.codigo_da_contratacao""")

insert = cur_dest.prep("""
INSERT INTO CONTRATOSADITAMENTO (CONTRATO, DTLAN, dataencerramento, valor, descricao, datainsc, TIPOHIST, TIPOALT, termo) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

for row in cur_orig:
    try:
        cur_dest.execute(insert, (
            row['codigo'],
            row['dtlan'],
            row['dataencerramento'],
            row['valor'],
            row['descricao'][:240] if row['descricao'] else None,
            row['datainsc'],
            row['tipohist'],
            row['tipoalt'],
            row['termo']
        ))
    except Exception as e:
        print(e)
        break
commit()